In [ ]:
#task1
import heapq

PIECE_VALUES = {'wp': 10, 'wn': 30, 'wb': 30, 'wr': 50, 'wq': 90, 'wk': 900,
    'bp': -10, 'bn': -30, 'bb': -30, 'br': -50, 'bq': -90, 'bk': -900,
    '--': 0}

def evaluate_board(board):
    score = 0
    for row in board:
        for piece in row:
            score += PIECE_VALUES.get(piece, 0)
    return score

def get_legal_moves(board, color):
    moves = []
    for r in range(8):
        for c in range(8):
            piece = board[r][c]
            if piece != '--' and piece[0] == color:
                directions = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]
                for dr, dc in directions:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < 8 and 0 <= nc < 8:
                        target = board[nr][nc]
                        if target == '--' or target[0] != color:
                            moves.append(((r, c), (nr, nc)))
    return moves

def apply_move(board, move):
    new_board = [row[:] for row in board]
    (r1, c1), (r2, c2) = move
    new_board[r2][c2] = new_board[r1][c1]
    new_board[r1][c1] = '--'
    return new_board

def beam_search(board, depth, width, color):
    beam = [(0, board, [])]
    
    for d in range(depth):
        candidates = []
        current_color = 'w' if (color == 'w' and d % 2 == 0) or (color == 'b' and d % 2 != 0) else 'b'
        
        for _, current_board, path in beam:
            moves = get_legal_moves(current_board, current_color)
            for move in moves:
                next_board = apply_move(current_board, move)
                next_score = evaluate_board(next_board)
                
                if color == 'b':
                    next_score *= -1
                
                candidates.append((next_score, next_board, path + [move]))
        
        if not candidates:
            break
            
        beam = heapq.nlargest(width, candidates, key=lambda x: x[0])
        
    return (beam[0][2], beam[0][0]) if beam else (None, 0)

initial_board = [
    ['br', 'bn', 'bb', 'bq', 'bk', 'bb', 'bn', 'br'],
    ['bp', 'bp', 'bp', 'bp', 'bp', 'bp', 'bp', 'bp'],
    ['--', '--', '--', '--', '--', '--', '--', '--'],
    ['--', '--', '--', '--', '--', '--', '--', '--'],
    ['--', '--', '--', '--', '--', '--', '--', '--'],
    ['--', '--', '--', '--', '--', '--', '--', '--'],
    ['wp', 'wp', 'wp', 'wp', 'wp', 'wp', 'wp', 'wp'],
    ['wr', 'wn', 'wb', 'wq', 'wk', 'wb', 'wn', 'wr']
]

best_sequence, final_score = beam_search(initial_board, depth=3, width=3, color='w')

print(f"Best Move Sequence: {best_sequence}")
print(f"Final Heuristic Score: {final_score}")

Best Move Sequence: [((6, 0), (5, 0)), ((1, 0), (2, 0)), ((5, 0), (4, 0))]
Final Heuristic Score: 0


In [ ]:
#task2
import math
import random

def calculate_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def total_route_distance(route):
    dist = 0
    for i in range(len(route) - 1):
        dist += calculate_distance(route[i], route[i+1])
    dist += calculate_distance(route[-1], route[0])
    return dist

def hill_climbing_tsp(locations, iterations=10000):
    current_route = list(locations)
    random.shuffle(current_route)
    current_distance = total_route_distance(current_route)

    for _ in range(iterations):
        i, j = random.sample(range(len(current_route)), 2)
        
        new_route = current_route[:]
        new_route[i], new_route[j] = new_route[j], new_route[i]
        
        new_distance = total_route_distance(new_route)

        if new_distance < current_distance:
            current_route = new_route
            current_distance = new_distance

    return current_route, current_distance

delivery_points = [(0, 0), (1, 5), (2, 2), (3, 3), (5, 1), (6, 4)]
best_route, min_distance = hill_climbing_tsp(delivery_points,3000)

print(f"Optimized Route: {best_route}")
print(f"Total Distance: {min_distance}")

Optimized Route: [(6, 4), (1, 5), (0, 0), (2, 2), (3, 3), (5, 1)]
Total Distance: 20.431384499219423


In [ ]:
#task3
import math
import random

cities = [(10, 20), (45, 80), (30, 10), (90, 50), (15, 75),(60, 30), (75, 90), (5, 40), (50, 55), (85, 10)]
n = len(cities)

def get_fitness(individual):
    d = 0
    for i in range(n):
        p1, p2 = cities[individual[i]], cities[individual[(i + 1) % n]]
        d += math.dist(p1, p2)
    return 1 / d 

def crossover(p1, p2):
    point = random.randint(1, n - 2)
    child = p1[:point] + p2[point:]

    missing = list(set(range(n)) - set(child))
    seen = set()
    for i in range(len(child)):
        if child[i] in seen:
            child[i] = missing.pop()
        else:
            seen.add(child[i])
    return child

def mutate(individual):
    idx1, idx2 = random.sample(range(n), 2)
    individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
    return individual

def solve_tsp():
    pop_size = 20
    population = [random.sample(range(n), n) for _ in range(pop_size)]
    
    for gen in range(200):
        scores = [get_fitness(ind) for ind in population]
        sorted_pop = [ind for _, ind in sorted(zip(scores, population), key=lambda x: x[0], reverse=True)]
        parents = sorted_pop[:pop_size // 2]
        new_pop = []

        while len(new_pop) < pop_size:
            p1, p2 = random.sample(parents, 2)
            new_pop.append(crossover(p1, p2))
    
        for i in range(pop_size):
            if random.random() < 0.1:
                new_pop[i] = mutate(new_pop[i])
        
        population = new_pop
    
    best = max(population, key=get_fitness)
    return best, 1/get_fitness(best)

route, dist = solve_tsp()
print(f"Shortest Route: {route}")
print(f"Total Distance: {dist}")

Shortest Route: [7, 0, 2, 5, 9, 3, 6, 1, 8, 4]
Total Distance: 327.9083616705137


In [ ]:
#task4
import heapq

tasks = [
    {'id': 'T1', 'time': 10, 'pri': 3},
    {'id': 'T2', 'time': 20, 'pri': 1},
    {'id': 'T3', 'time': 5,  'pri': 2},
    {'id': 'T4', 'time': 15, 'pri': 3},
    {'id': 'T5', 'time': 30, 'pri': 1}
]
num_processors = 3
k = 2

def beam_search_jobs():
    sorted_tasks = sorted(tasks, key=lambda x: x['pri'], reverse=True)
    beam = [(0, [0] * num_processors)]
    
    for task in sorted_tasks:
        candidates = []
        
        for _, loads in beam:
            for p in range(num_processors):
                new_loads = list(loads)
                new_loads[p] += task['time']
                h_score = max(new_loads)
                candidates.append((h_score, new_loads))
        
   
        beam = heapq.nsmallest(k, candidates, key=lambda x: x[0])
        
    return beam[0]

best_max, final_distribution = beam_search_jobs()
print(f"Max Load: {best_max}")
print(f"Final Processor Loads: {final_distribution}")

Max Load: 40
Final Processor Loads: [40, 15, 25]
